In [1]:
import os, sys
sys.path.append("../..")
from utilities import plot_wellbore, RogiDataset
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from plotly import express as px
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

data_path = "../../rogii-wellbore-geology-prediction"

# To Do
1. Data
    - Define what a sample should be.
    - Implement it
2. Model
3. Test


In [2]:
class RogiDataset():
    def __init__(self, path=Path("../../rogii-wellbore-geology-prediction/train",)):
        self.path = path
        self.well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in self.path.glob("*__horizontal_well.csv")
            }
        self.keys = list(self.well_files.keys())
        self._index=0

    def __getitem__(self,key:str|int):
        if isinstance(key, int):
            key = self.keys[key]
        well_data, typewell_data = pd.read_csv(self.well_files[key]["Well"]),pd.read_csv(self.well_files[key]["TypeWell"])
        well_data, typewell_data = well_data.set_index("MD").sort_index(), typewell_data.set_index("TVT").sort_index()
        pred_start_idx = well_data.index[(~well_data.TVT_input.isna()).sum()]
        last_known_idx = well_data.index[~well_data.TVT_input.isna()][-1]
        well_data["Known"] = well_data.index<=last_known_idx
        return well_data, typewell_data

    def __len__(self):
        return len(self.keys)
    
    def __iter__(self):
        self._index=0
        return self

    def __next__(self):
        if self._index>= len(self):
            raise StopIteration
        val = self[self._index]
        self._index+=1
        return val
        

In [176]:
rog = RogiDataset()


# Encoding the data
The model will take both the well and typewell logs.

## Well Log
The well log data, will be sent to the well log head of the model.
- It will be of shape (batch, channel, well_log_window_size)
    - well_log_window_size is a tunable hyperparameter. It should be large enough to capture enough GR variation to disambiguate the location from the typewell log.
- Each "sample" will be a window of gr data, anchored (at the start(left)) at any MD point along the wellbore.
- The spacing of the MD index will be 1 (the data default), and the gr log will have been interpolated to remove all nans.

## Typewell Log
The typewell log data, will be sent to the typewell log head of the model.
- shape: (batch, channel, typewell_log_window_size)
    - typewell_log_window_size is a tunable hyper parameter, this is the actual length of the window
    - The tvt span of the data within that window, will be set as another tunable hyper parameter: typewell_log_range
- Each sample, will be a window of the typewell log TVT, anchored in the middle (typewell_tvt_anchor), at the closest matching tvt
    - Since tvt index of the typewell is not always spread evenly, we need to reindex that window (bounds at )

In [259]:
class MatchingDataset(RogiDataset):
    """Create local well/typewell window pairs for TVT-difference training."""

    def __init__(
            self, path=Path("../../rogii-wellbore-geology-prediction/train"), 
            well_wnd_size=100, 
            typewell_wnd_size=200, typewell_tvt_interval=0.5,
            ):
        """Initialize the dataset and precompute how many windows each well contributes."""
        super().__init__(path)
        self.well_wnd_size = well_wnd_size
        self.typewell_wnd_size = typewell_wnd_size
        self.typewell_tvt_interval = typewell_tvt_interval
        self.typewell_wnd_tvt_range = self.typewell_tvt_interval*self.typewell_wnd_size/2

        well_num_idxs = []
        for i in tqdm(range(super().__len__())):
            w,t = super().__getitem__(i)
            well_num_idxs.append((~w.Known).sum()-(self.well_wnd_size-1))
        self.well_num_idxs = np.array(well_num_idxs, dtype=int)
        self.well_num_idxs_cumsum = self.well_num_idxs.cumsum()

    def __len__(self):
        """Return the total number of sliding windows across all wells."""
        return self.well_num_idxs.sum()
        
    def get_well_typewell_pair(self,idx) -> tuple[pd.DataFrame, pd.DataFrame]:
        """Load one raw well and its paired typewell by integer index."""
        return super().__getitem__(idx)
    
    def filter_srs(self, srs:pd.Series, wnd_size=3, mode="median") -> pd.Series:
        """Apply a light rolling filter while preserving the series length."""
        rll = srs.rolling(wnd_size, center=True,min_periods=1)
        if mode=="median":
            return rll.median()
        elif mode=="mean":
            return rll.mean()
        else:
            return srs
    
    def _process_typewell(self, typewell:pd.DataFrame)->pd.Series:
        typewell_gr = typewell.GR
        grid = np.arange(typewell_gr.index[0], typewell_gr.index[-1]+self.typewell_tvt_interval, self.typewell_tvt_interval)
        typewell_gr = pd.Series(
            np.interp(grid, typewell_gr.index.to_numpy(dtype=float), typewell_gr.to_numpy(dtype=float)),
            index=grid,
            name="GR"
            )
        typewell_gr = self.filter_srs(typewell_gr)
        return typewell_gr
    
    def _process_well(self, well:pd.DataFrame)->pd.DataFrame:
        """Filter the well GR and keep only the unknown interval used for training."""
        well = well.copy()
        well["GR"] = self.filter_srs(well.GR).interpolate()
        well = well.loc[~well.Known]
        return well
    
    def _get_well_selection(self,key):
        """Map a global sample index to a specific well and local window offset."""
        selected_well = int(np.searchsorted(self.well_num_idxs_cumsum, key, side="right"))
        selected_well_idx = int(key-self.well_num_idxs[:selected_well].sum())
        return selected_well, selected_well_idx

    def _get_typewell_window(self, typewell_gr: pd.Series, anchor_tvt: float) -> pd.Series:
        """Extract a fixed-width typewell window centered as closely as possible on the anchor TVT."""
        typewell_anchor = int(np.argmin(np.abs(typewell_gr.index - anchor_tvt)))
        half_window = self.typewell_wnd_size // 2
        start = min(max(typewell_anchor - half_window, 0), typewell_gr.shape[0] - self.typewell_wnd_size)
        stop = start + self.typewell_wnd_size
        return typewell_gr.iloc[start:stop]

    def __getitem__(self, key:int):
        """Return a well GR window, the aligned typewell window, and TVT offsets from the window start."""
        total_samples = len(self)
        if key<0:
            key = total_samples+key
        if key < 0 or key >= total_samples:
            raise IndexError("MatchingDataset index out of range")
        selected_well, selected_well_idx = self._get_well_selection(key)
        well, typewell = self.get_well_typewell_pair(selected_well)
        well, typewell_gr = self._process_well(well), self._process_typewell(typewell)
        well_wnd = well.iloc[selected_well_idx:selected_well_idx+self.well_wnd_size]
        anchor_tvt = well_wnd.TVT.iloc[0]
        typewell_wnd = self._get_typewell_window(typewell_gr, anchor_tvt)
        well_wnd_gr = well_wnd.GR
        tvt_diff = well_wnd.TVT - anchor_tvt
        return well_wnd_gr, typewell_wnd, tvt_diff
    
    
self = MatchingDataset()

  0%|          | 0/773 [00:00<?, ?it/s]

In [258]:
for i in tqdm(np.linspace(0,len(self)-1,1400)):
    w,t,td = self[i]
    assert (not w.isna().any()) and (not t.isna().any())

  0%|          | 0/1400 [00:00<?, ?it/s]